# LeetCode 562 — Longest Line of Consecutive One in Matrix bằng Python

Link đề: https://leetcode.com/problems/longest-line-of-consecutive-one-in-matrix/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Matrix

Skeleton LeetCode:

```python
class Solution:
    def longestLine(self, mat: List[List[int]]) -> int:
        ...
```

Ghi chú: đây là bài **Premium** (cần tài khoản trả phí để xem trên LeetCode). Đề trong notebook này được viết theo hiểu biết về bài (có đối chiếu bản mô tả tiếng Anh), nên nếu bạn có tài khoản Premium hãy **đối chiếu lại đề gốc** về câu chữ, ràng buộc và ví dụ.

Notebook đi từ cách "đi thẳng từ mọi ô 1" `O(mn(m+n))`, tới "chỉ bắt đầu ở đầu mỗi đường" `O(mn)` thời gian `O(1)` bộ nhớ, tới **quy hoạch động (DP) 4 hướng**, DP cuộn hàng `O(n)` bộ nhớ, và cách gom theo "đường thẳng" bằng chuỗi.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong ma trận nhị phân `m x n`, tìm độ dài **dãy các số 1 liên tiếp dài nhất** nằm trên một **đường thẳng** theo một trong **4 hướng**: ngang, dọc, chéo chính, chéo phụ.
2. Nắm công cụ Python/ma trận: `mat[i][j]`, véc-tơ hướng `(di, dj)`, kiểm tra biên `0 <= x < m`, đường chéo theo `i - j` và `i + j`, tạo bảng 2D đúng cách, `max` nhiều đối số, `str.split`.
3. Đi từ **thử mọi ô × 4 hướng** → **chỉ bắt đầu từ đầu mỗi đường** → **DP bốn bảng** → **DP cuộn hàng** → **gom theo đường thẳng bằng chuỗi**.
4. Hiểu mấu chốt DP: độ dài dãy 1 kết thúc tại `(i, j)` theo hướng `d` bằng **độ dài tại ô đứng trước theo hướng `d`, cộng 1** (nếu ô hiện tại là 1), ngược lại bằng 0.
5. Tránh lỗi kinh điển: quên chéo phụ, chỉ số âm "quay vòng" của Python, quên reset khi gặp số 0, tạo bảng 2D bị **dùng chung hàng** (`[[0]*n]*m`), và nhầm `m`/`n`.

## 2. Đề bài & ví dụ

Cho ma trận nhị phân `mat` kích thước `m x n` (mỗi phần tử là `0` hoặc `1`). Hãy trả về **độ dài của đường thẳng dài nhất gồm các số 1 liên tiếp** trong ma trận.

Đường thẳng có thể là **ngang**, **dọc**, **chéo chính** (từ trên-trái xuống dưới-phải) hoặc **chéo phụ** (từ trên-phải xuống dưới-trái).

**Ví dụ 1**

```
0 1 1 0
0 1 1 0
0 0 0 1
```

- Output: `3` (đường chéo chính đi qua `(0,1) -> (1,2) -> (2,3)`).

**Ví dụ 2**

```
1 1 1 1
0 1 1 0
0 0 0 1
```

- Output: `4` (hàng đầu tiên toàn số 1).

**Ràng buộc (theo bài gốc, cần đối chiếu lại nếu có Premium):**

- `m == mat.length`, `n == mat[i].length`
- `1 <= m, n <= 10^4`
- `1 <= m * n <= 10^4`
- `mat[i][j]` là `0` hoặc `1`

Lưu ý `m` và `n` có thể rất lệch nhau (ví dụ `1 x 10000` hay `10000 x 1`), nhưng tích tối đa chỉ `10^4` nên duyệt cả ma trận vài lần là rất nhẹ. Ta cần thuật toán `O(m*n)`.

In [1]:
from typing import List
from itertools import groupby
import random

ex1 = [[0, 1, 1, 0],
       [0, 1, 1, 0],
       [0, 0, 0, 1]]
ex2 = [[1, 1, 1, 1],
       [0, 1, 1, 0],
       [0, 0, 0, 1]]
examples = [(ex1, 3), (ex2, 4)]


def show(mat):
    for row in mat:
        print(" ".join(str(v) for v in row))


for mat, expected in examples:
    print(f"m={len(mat)} n={len(mat[0])} | kỳ vọng {expected}")
    show(mat)
    print()

m=3 n=4 | kỳ vọng 3
0 1 1 0
0 1 1 0
0 0 0 1

m=3 n=4 | kỳ vọng 4
1 1 1 1
0 1 1 0
0 0 0 1



## 3. Kiến thức nền

### 3.1. Ma trận là "list của list": `mat[i][j]`

`mat[i]` là **hàng** thứ `i`, `mat[i][j]` là ô ở hàng `i`, cột `j` (đếm từ 0). Số hàng `m = len(mat)`, số cột `n = len(mat[0])`. Hàng ngang đi theo `j`, cột dọc đi theo `i`.

In [2]:
mat = ex1
m, n = len(mat), len(mat[0])
print("m =", m, "| n =", n)
print("mat[1] =", mat[1], "| mat[1][2] =", mat[1][2])
print("cột 1 =", [mat[i][1] for i in range(m)])
for i in range(m):
    for j in range(n):
        if mat[i][j] == 1:
            print(f"ô 1 tại ({i},{j})", end="; ")
print()

m = 3 | n = 4
mat[1] = [0, 1, 1, 0] | mat[1][2] = 1
cột 1 = [1, 1, 0]
ô 1 tại (0,1); ô 1 tại (0,2); ô 1 tại (1,1); ô 1 tại (1,2); ô 1 tại (2,3); 


### 3.2. Bốn hướng = bốn véc-tơ `(di, dj)`

Đi một bước theo hướng `(di, dj)` nghĩa là `(i, j) -> (i + di, j + dj)`:

| hướng | `(di, dj)` | cách đi |
|---|---|---|
| ngang | `(0, 1)` | sang phải |
| dọc | `(1, 0)` | xuống dưới |
| chéo chính | `(1, 1)` | xuống dưới-phải |
| chéo phụ | `(1, -1)` | xuống dưới-trái |

Chỉ cần xét **một chiều** của mỗi đường (ví dụ luôn đi xuống/sang phải), vì dãy 1 liên tiếp đọc xuôi hay ngược đều dài như nhau. Trước khi đọc ô mới ta phải **kiểm tra biên**: `0 <= x < m` và `0 <= y < n`.

In [3]:
DIRS = [(0, 1), (1, 0), (1, 1), (1, -1)]
NAMES = ["ngang", "dọc", "chéo chính", "chéo phụ"]

def in_bounds(x, y, m, n):
    return 0 <= x < m and 0 <= y < n

i, j = 0, 1
for (di, dj), name in zip(DIRS, NAMES):
    path = []
    x, y = i, j
    while in_bounds(x, y, 3, 4):
        path.append((x, y))
        x, y = x + di, y + dj
    print(f"{name:11s} từ {(i, j)}: {path}")

ngang       từ (0, 1): [(0, 1), (0, 2), (0, 3)]
dọc         từ (0, 1): [(0, 1), (1, 1), (2, 1)]
chéo chính  từ (0, 1): [(0, 1), (1, 2), (2, 3)]
chéo phụ    từ (0, 1): [(0, 1), (1, 0)]


### 3.3. Nhận diện đường chéo bằng `i - j` và `i + j`

- Các ô cùng **một chéo chính** có **`i - j` bằng nhau** (đi xuống-phải thì `i` và `j` cùng tăng 1).
- Các ô cùng **một chéo phụ** có **`i + j` bằng nhau** (đi xuống-trái thì `i` tăng, `j` giảm).

Nhờ đó ta có thể **gom ô theo "mã đường"**: hàng `i`, cột `j`, chéo `i - j`, chéo phụ `i + j`.

In [4]:
m, n = 3, 4
diag = {}
anti = {}
for i in range(m):
    for j in range(n):
        diag.setdefault(i - j, []).append((i, j))
        anti.setdefault(i + j, []).append((i, j))
print("chéo chính (i - j):")
for key in sorted(diag):
    print(f"  {key:>2}: {diag[key]}")
print("chéo phụ (i + j):")
for key in sorted(anti):
    print(f"  {key:>2}: {anti[key]}")
print("số chéo chính:", len(diag), "= m + n - 1 =", m + n - 1)

chéo chính (i - j):
  -3: [(0, 3)]
  -2: [(0, 2), (1, 3)]
  -1: [(0, 1), (1, 2), (2, 3)]
   0: [(0, 0), (1, 1), (2, 2)]
   1: [(1, 0), (2, 1)]
   2: [(2, 0)]
chéo phụ (i + j):
   0: [(0, 0)]
   1: [(0, 1), (1, 0)]
   2: [(0, 2), (1, 1), (2, 0)]
   3: [(0, 3), (1, 2), (2, 1)]
   4: [(1, 3), (2, 2)]
   5: [(2, 3)]
số chéo chính: 6 = m + n - 1 = 6


### 3.4. Padding (viền 0) và bẫy chỉ số âm của Python

Ở ô biên, "ô đứng trước" có thể **nằm ngoài** ma trận. Có hai cách xử lý:

1. Viết `if` kiểm tra biên mỗi lần.
2. **Thêm viền 0** quanh bảng DP (kích thước `(m+2) x (n+2)`), để ô ngoài biên luôn đọc ra `0`, không cần `if`.

Bẫy riêng của Python: chỉ số **âm là hợp lệ** — `a[-1]` là phần tử **cuối cùng**, không báo lỗi! Nên nếu bạn viết `dp[i-1][j-1]` khi `j = 0`, bạn lỡ đọc `dp[i-1][-1]` (cột cuối hàng trên) mà không có cảnh báo nào. Còn chỉ số vượt phải như `a[n]` thì báo `IndexError`. Padding giải quyết cả hai.

In [5]:
row = [10, 20, 30]
print("row[-1] =", row[-1], "(phần tử cuối, KHÔNG phải lỗi)")
try:
    row[3]
except IndexError as e:
    print("row[3] -> IndexError:", e)

m, n = 2, 3
padded = [[0] * (n + 2) for _ in range(m + 2)]
print("kích thước bảng có viền:", len(padded), "x", len(padded[0]))
# ô (i, j) của ma trận gốc nằm ở padded[i + 1][j + 1]; hàng/cột 0 và m+1/n+1 là viền

row[-1] = 30 (phần tử cuối, KHÔNG phải lỗi)
row[3] -> IndexError: list index out of range
kích thước bảng có viền: 4 x 5


### 3.5. Tạo bảng 2D đúng cách

`[[0] * n for _ in range(m)]` tạo `m` list **khác nhau**. Còn `[[0] * n] * m` tạo `m` tham chiếu tới **cùng một list** — sửa một hàng là sửa tất cả. Đây là lỗi rất hay gặp; ta sẽ gặp lại ở mục 13.

In [6]:
good = [[0] * 3 for _ in range(2)]
bad = [[0] * 3] * 2
good[0][1] = 7
bad[0][1] = 7
print("đúng:", good)
print("sai :", bad, "| hai hàng là cùng một đối tượng?", bad[0] is bad[1])

đúng: [[0, 7, 0], [0, 0, 0]]
sai : [[0, 7, 0], [0, 7, 0]] | hai hàng là cùng một đối tượng? True


## 4. Ý tưởng nền tảng / trực giác

Hãy nhìn ví dụ 1 và tìm tay đường dài nhất:

```
   j: 0 1 2 3
i=0:  0 1 1 0
i=1:  0 1 1 0
i=2:  0 0 0 1
```

- **Ngang**: hàng 0 có `1 1` (dài 2), hàng 1 có `1 1` (dài 2), hàng 2 có `1` → tối đa 2.
- **Dọc**: cột 1 có `1 1` (2), cột 2 có `1 1` (2), cột 3 có `1` → tối đa 2.
- **Chéo chính** (`i - j` cố định): `i - j = -1` gồm `(0,1),(1,2),(2,3)` toàn `1` → **3**.
- **Chéo phụ** (`i + j` cố định): `i + j = 2` gồm `(0,2),(1,1),(2,0)` = `1,1,0` → 2.

Kết quả `max = 3`. Nhận xét quan trọng: mỗi đường thẳng là một **dãy 1 chiều**, và bài toán trên mỗi dãy chỉ là "tìm đoạn 1 liên tiếp dài nhất" (như LeetCode 485). Vậy ta chỉ cần làm việc đó cho **mọi đường thẳng của cả 4 loại**.

In [7]:
def lines_of(mat):
    """Trả về mọi đường thẳng (4 loại) dưới dạng danh sách giá trị, để quan sát."""
    m, n = len(mat), len(mat[0])
    res = {"ngang": [], "dọc": [], "chéo chính": {}, "chéo phụ": {}}
    for i in range(m):
        res["ngang"].append([mat[i][j] for j in range(n)])
    for j in range(n):
        res["dọc"].append([mat[i][j] for i in range(m)])
    for i in range(m):
        for j in range(n):
            res["chéo chính"].setdefault(i - j, []).append(mat[i][j])
            res["chéo phụ"].setdefault(i + j, []).append(mat[i][j])
    return res


def longest_run_1d(seq):
    best = cur = 0
    for v in seq:
        cur = cur + 1 if v == 1 else 0
        best = max(best, cur)
    return best


L = lines_of(ex1)
for name in ("ngang", "dọc"):
    print(name, [longest_run_1d(s) for s in L[name]])
for name in ("chéo chính", "chéo phụ"):
    print(name, {k: longest_run_1d(s) for k, s in sorted(L[name].items())})

ngang [2, 2, 1]
dọc [0, 2, 2, 1]
chéo chính {-3: 0, -2: 1, -1: 3, 0: 1, 1: 0, 2: 0}
chéo phụ {0: 0, 1: 1, 2: 2, 3: 1, 4: 0, 5: 1}


## 5. Cách 1 — Brute Force: từ mỗi ô 1, đi thẳng theo 4 hướng

### Ý tưởng

Với **mỗi ô có giá trị 1**, và với **mỗi hướng** trong `DIRS`, đi liên tiếp theo hướng đó cho tới khi gặp ô `0` hoặc ra khỏi biên, đếm số bước. Lấy `max` của mọi lần đi.

Cách này luôn đúng vì đường dài nhất chắc chắn được "đi qua" khi ta xuất phát đúng từ **đầu đường**. Ta dùng nó làm **oracle** cho mọi test sau này.

In [8]:
def line_brute(mat):
    m, n = len(mat), len(mat[0])
    best = 0
    for i in range(m):
        for j in range(n):
            if mat[i][j] != 1:
                continue
            for di, dj in DIRS:
                length = 0
                x, y = i, j
                while 0 <= x < m and 0 <= y < n and mat[x][y] == 1:
                    length += 1
                    x += di
                    y += dj
                best = max(best, length)
    return best


for mat, expected in examples:
    print(line_brute(mat), "| kỳ vọng", expected)
    assert line_brute(mat) == expected

3 | kỳ vọng 3
4 | kỳ vọng 4


### Độ phức tạp và nút thắt

- Mỗi ô 1 × 4 hướng, mỗi lần đi tối đa `max(m, n)` bước → `O(m * n * max(m, n))` thời gian (tối đa cỡ `10^8` khi mảng `1 x 10000` toàn 1; trong Python là quá chậm). Bộ nhớ `O(1)`.

Nút thắt: ta **đi lại những ô đã đi**. Ví dụ trên một hàng `1 1 1 1`, xuất phát từ ô thứ nhất đi 4 bước, từ ô thứ hai đi 3 bước, ... trong khi chỉ cần xuất phát từ ô đầu tiên. Hai cải tiến tự nhiên: (a) **chỉ xuất phát ở đầu mỗi đường** (cách 2), (b) **nhớ lại độ dài đã tính** (DP, cách 3).

## 6. Ý tưởng thuật toán / bước đột phá

Gọi `f_d[i][j]` là **độ dài dãy 1 liên tiếp kết thúc tại `(i, j)` theo hướng `d`** (đi từ ô đứng trước tới ô `(i, j)`). Ô đứng trước theo từng hướng:

| hướng | ô đứng trước của `(i, j)` |
|---|---|
| ngang | `(i, j-1)` |
| dọc | `(i-1, j)` |
| chéo chính | `(i-1, j-1)` |
| chéo phụ | `(i-1, j+1)` |

Công thức:

- nếu `mat[i][j] == 0` thì `f_d[i][j] = 0`;
- nếu `mat[i][j] == 1` thì `f_d[i][j] = f_d[ô đứng trước][...] + 1`.

Đáp án là `max` của mọi `f_d[i][j]`. Vì khi duyệt theo thứ tự hàng (trên xuống dưới, trái sang phải) thì cả bốn ô đứng trước **đã được tính xong** — đây là DP tuyến tính `O(m*n)`.

Phát hiện phụ: bài hóa ra không cần tới bảng, vì mỗi `f_d` chỉ phụ thuộc **hàng trên** và **ô bên trái** → có thể cuộn hàng (cách 4).

## 7. Các cách tiếp theo

Tổng cộng 5 cách: (1) từ mọi ô × 4 hướng; (2) chỉ bắt đầu từ đầu đường; (3) DP bốn bảng; (4) DP cuộn hàng; (5) gom theo đường thẳng bằng chuỗi.

### 7.1. Cách 2 — Chỉ xuất phát từ ĐẦU mỗi đường, `O(mn)` thời gian, `O(1)` bộ nhớ

**Giải thích từng đoạn.**

1. Với mỗi ô 1 và mỗi hướng `(di, dj)`, xem **ô đứng trước** `(i - di, j - dj)`.
2. Nếu ô đứng trước nằm ngoài biên hoặc bằng 0 thì `(i, j)` là **đầu của một dãy** — chỉ khi đó mới đi thẳng để đếm độ dài.
3. Nếu ô đứng trước là 1 thì bỏ qua, vì dãy này sẽ được đếm từ đầu của nó.

Mỗi ô nằm trong đúng một dãy theo mỗi hướng và chỉ bị "đi qua" một lần khi dãy đó được đếm → tổng `O(4 * m * n)` bước.

In [9]:
def line_run_start(mat):
    m, n = len(mat), len(mat[0])
    best = 0
    for i in range(m):
        for j in range(n):
            if mat[i][j] != 1:
                continue
            for di, dj in DIRS:
                px, py = i - di, j - dj                      # ô đứng trước
                if 0 <= px < m and 0 <= py < n and mat[px][py] == 1:
                    continue                                 # không phải đầu dãy
                length = 0
                x, y = i, j
                while 0 <= x < m and 0 <= y < n and mat[x][y] == 1:
                    length += 1
                    x += di
                    y += dj
                best = max(best, length)
    return best


for mat, expected in examples:
    print(line_run_start(mat), "| kỳ vọng", expected)
    assert line_run_start(mat) == expected

3 | kỳ vọng 3
4 | kỳ vọng 4


**Code nộp LeetCode (cách 2):**

```python
class Solution:
    def longestLine(self, mat: List[List[int]]) -> int:
        m, n = len(mat), len(mat[0])
        dirs = [(0, 1), (1, 0), (1, 1), (1, -1)]
        best = 0
        for i in range(m):
            for j in range(n):
                if mat[i][j] != 1:
                    continue
                for di, dj in dirs:
                    px, py = i - di, j - dj
                    if 0 <= px < m and 0 <= py < n and mat[px][py] == 1:
                        continue
                    length, x, y = 0, i, j
                    while 0 <= x < m and 0 <= y < n and mat[x][y] == 1:
                        length += 1
                        x += di
                        y += dj
                    best = max(best, length)
        return best
```

**Độ phức tạp:** Time `O(m * n)` (mỗi ô bị đi qua tối đa 4 lần, một lần cho mỗi hướng), Space `O(1)`.

### 7.2. Cách 3 — DP với bốn bảng có viền `O(mn)`

**Giải thích từng đoạn.**

1. Tạo 4 bảng `hor, ver, dia, ant` kích thước `(m+2) x (n+2)` toàn 0 (viền 0 giúp mọi ô đứng trước đều đọc được).
2. Duyệt `i = 1..m`, `j = 1..n` (ô `mat[i-1][j-1]` ứng với `bảng[i][j]`).
3. Nếu ô là 1: `hor[i][j] = hor[i][j-1] + 1`, `ver[i][j] = ver[i-1][j] + 1`, `dia[i][j] = dia[i-1][j-1] + 1`, `ant[i][j] = ant[i-1][j+1] + 1`.
4. Cập nhật `best = max(best, bốn giá trị đó)`. Ô là 0 thì để nguyên 0.

Viền cột `n+1` chính là chỗ `ant[i-1][j+1]` đọc khi `j = n`; viền cột 0 là chỗ `dia[i-1][j-1]` đọc khi `j = 1`.

In [10]:
def line_dp4(mat):
    m, n = len(mat), len(mat[0])
    hor = [[0] * (n + 2) for _ in range(m + 2)]
    ver = [[0] * (n + 2) for _ in range(m + 2)]
    dia = [[0] * (n + 2) for _ in range(m + 2)]
    ant = [[0] * (n + 2) for _ in range(m + 2)]
    best = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if mat[i - 1][j - 1]:
                hor[i][j] = hor[i][j - 1] + 1
                ver[i][j] = ver[i - 1][j] + 1
                dia[i][j] = dia[i - 1][j - 1] + 1
                ant[i][j] = ant[i - 1][j + 1] + 1
                best = max(best, hor[i][j], ver[i][j], dia[i][j], ant[i][j])
    return best


for mat, expected in examples:
    print(line_dp4(mat), "| kỳ vọng", expected)
    assert line_dp4(mat) == expected

3 | kỳ vọng 3
4 | kỳ vọng 4


**Code nộp LeetCode (cách 3):**

```python
class Solution:
    def longestLine(self, mat: List[List[int]]) -> int:
        m, n = len(mat), len(mat[0])
        hor = [[0] * (n + 2) for _ in range(m + 2)]
        ver = [[0] * (n + 2) for _ in range(m + 2)]
        dia = [[0] * (n + 2) for _ in range(m + 2)]
        ant = [[0] * (n + 2) for _ in range(m + 2)]
        best = 0
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if mat[i - 1][j - 1]:
                    hor[i][j] = hor[i][j - 1] + 1
                    ver[i][j] = ver[i - 1][j] + 1
                    dia[i][j] = dia[i - 1][j - 1] + 1
                    ant[i][j] = ant[i - 1][j + 1] + 1
                    best = max(best, hor[i][j], ver[i][j], dia[i][j], ant[i][j])
        return best
```

**Độ phức tạp:** Time `O(m * n)`, Space `O(m * n)` (4 bảng).

### 7.3. Cách 4 — DP cuộn hàng, bộ nhớ `O(n)`

**Ý tưởng.** Ở hàng `i`, các bảng chỉ đọc **hàng `i-1`** (cho dọc, chéo chính, chéo phụ) và **ô bên trái cùng hàng** (cho ngang). Vậy ta chỉ giữ:

- ba mảng của hàng trước: `up` (dọc), `dg` (chéo chính), `ad` (chéo phụ), mỗi mảng độ dài `n+2` có viền;
- một số nguyên `left` cho hướng ngang, **reset về 0 khi gặp ô 0**.

Sau mỗi hàng ta "đẩy" hàng hiện tại thành hàng trước.

In [11]:
def line_rolling(mat):
    m, n = len(mat), len(mat[0])
    up = [0] * (n + 2)          # dọc ở hàng trước
    dg = [0] * (n + 2)          # chéo chính ở hàng trước
    ad = [0] * (n + 2)          # chéo phụ ở hàng trước
    best = 0
    for i in range(m):
        cu = [0] * (n + 2)
        cd = [0] * (n + 2)
        ca = [0] * (n + 2)
        left = 0                # ngang: chuỗi 1 liên tiếp kết thúc ở ô trước
        for j in range(1, n + 1):
            if mat[i][j - 1]:
                left += 1
                cu[j] = up[j] + 1
                cd[j] = dg[j - 1] + 1
                ca[j] = ad[j + 1] + 1
                best = max(best, left, cu[j], cd[j], ca[j])
            else:
                left = 0        # QUAN TRỌNG: gặp 0 thì cắt chuỗi ngang
        up, dg, ad = cu, cd, ca
    return best


for mat, expected in examples:
    print(line_rolling(mat), "| kỳ vọng", expected)
    assert line_rolling(mat) == expected

3 | kỳ vọng 3
4 | kỳ vọng 4


**Code nộp LeetCode (cách 4):**

```python
class Solution:
    def longestLine(self, mat: List[List[int]]) -> int:
        m, n = len(mat), len(mat[0])
        up = [0] * (n + 2)
        dg = [0] * (n + 2)
        ad = [0] * (n + 2)
        best = 0
        for i in range(m):
            cu = [0] * (n + 2)
            cd = [0] * (n + 2)
            ca = [0] * (n + 2)
            left = 0
            for j in range(1, n + 1):
                if mat[i][j - 1]:
                    left += 1
                    cu[j] = up[j] + 1
                    cd[j] = dg[j - 1] + 1
                    ca[j] = ad[j + 1] + 1
                    best = max(best, left, cu[j], cd[j], ca[j])
                else:
                    left = 0
            up, dg, ad = cu, cd, ca
        return best
```

**Độ phức tạp:** Time `O(m * n)`, Space `O(n)`. (Nếu `n` lớn hơn `m` nhiều, như `1 x 10000`, có thể **chuyển vị** ma trận để cột ít hơn; nhưng chuyển vị đổi chéo phụ thành chéo phụ và chéo chính thành chéo chính, nên đáp án không đổi. Ở đây ta không cần, vì `O(n) <= O(10^4)` đã nhỏ.)

### 7.4. Cách 5 — Gom theo đường thẳng + cắt chuỗi

**Ý tưởng khác hẳn DP.** Gán cho mỗi ô 4 "mã đường": `(hàng i)`, `(cột j)`, `(chéo i - j)`, `(chéo phụ i + j)`. Với mỗi loại, gom các ký tự `'0'/'1'` của những ô cùng mã **theo thứ tự duyệt** thành một chuỗi, rồi dùng `chuỗi.split('0')` để cắt thành các cụm 1; độ dài cụm dài nhất là đáp án của đường đó.

Ví dụ: đường `"0110111"` → `split('0')` → `['', '11', '111']` → dài nhất `3`.

Cách này **dễ chứng minh đúng** nhất (nó chỉ là "đoạn 1 liên tiếp dài nhất" trên từng dãy) và độc lập hẳn với DP, nên rất hợp để đối chiếu.

In [12]:
def line_group(mat):
    m, n = len(mat), len(mat[0])
    rows = [[] for _ in range(m)]
    cols = [[] for _ in range(n)]
    diag, anti = {}, {}
    for i in range(m):
        for j in range(n):
            ch = "1" if mat[i][j] else "0"
            rows[i].append(ch)
            cols[j].append(ch)
            diag.setdefault(i - j, []).append(ch)
            anti.setdefault(i + j, []).append(ch)
    best = 0
    groups = rows + cols + list(diag.values()) + list(anti.values())
    for g in groups:
        s = "".join(g)
        best = max(best, max(len(part) for part in s.split("0")))
    return best


print("ví dụ cắt chuỗi:", "0110111".split("0"))
for mat, expected in examples:
    print(line_group(mat), "| kỳ vọng", expected)
    assert line_group(mat) == expected

ví dụ cắt chuỗi: ['', '11', '111']
3 | kỳ vọng 3
4 | kỳ vọng 4


**Code nộp LeetCode (cách 5):**

```python
class Solution:
    def longestLine(self, mat: List[List[int]]) -> int:
        m, n = len(mat), len(mat[0])
        rows = [[] for _ in range(m)]
        cols = [[] for _ in range(n)]
        diag, anti = {}, {}
        for i in range(m):
            for j in range(n):
                ch = "1" if mat[i][j] else "0"
                rows[i].append(ch)
                cols[j].append(ch)
                diag.setdefault(i - j, []).append(ch)
                anti.setdefault(i + j, []).append(ch)
        best = 0
        for g in rows + cols + list(diag.values()) + list(anti.values()):
            best = max(best, max(len(p) for p in "".join(g).split("0")))
        return best
```

**Độ phức tạp:** Time `O(m * n)` (mỗi ô vào 4 nhóm), Space `O(m * n)` (lưu ký tự của mọi nhóm).

Năm cách cho cùng đáp án; mục 12 sẽ đối chiếu chúng bằng `assert` trên rất nhiều ma trận.

## 8. TRACE từng bước

Trace cách 3 (DP bốn bảng) trên ví dụ 1. Ta chỉ in các ô có giá trị 1 vì ô 0 luôn giữ bốn giá trị 0.

Bảng kết quả tay (tọa độ trong ma trận gốc, thứ tự `ngang, dọc, chéo chính, chéo phụ`):

| ô `(i,j)` | ngang | dọc | chéo chính | chéo phụ | `best` sau ô này |
|---|---|---|---|---|---|
| `(0,1)` | 1 | 1 | 1 | 1 | 1 |
| `(0,2)` | 2 | 1 | 1 | 1 | 2 |
| `(1,1)` | 1 | 2 | 1 | 2 | 2 |
| `(1,2)` | 2 | 2 | 2 | 1 | 2 |
| `(2,3)` | 1 | 1 | **3** | 1 | **3** |

Giải thích hai ô đáng chú ý:

- `(1,1)`: dọc = dọc của `(0,1)` + 1 = 2; chéo phụ = chéo phụ của `(0,2)` + 1 = 2 (đường `(0,2),(1,1)`).
- `(2,3)`: chéo chính = chéo chính của `(1,2)` + 1 = 3 (đường `(0,1),(1,2),(2,3)`).

Hàm `trace_dp` dưới đây in đúng các dòng này và in bốn bảng cuối cùng.

In [13]:
def trace_dp(mat):
    m, n = len(mat), len(mat[0])
    hor = [[0] * (n + 2) for _ in range(m + 2)]
    ver = [[0] * (n + 2) for _ in range(m + 2)]
    dia = [[0] * (n + 2) for _ in range(m + 2)]
    ant = [[0] * (n + 2) for _ in range(m + 2)]
    best = 0
    print("thứ tự cột: (ngang, dọc, chéo chính, chéo phụ)")
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if not mat[i - 1][j - 1]:
                continue
            hor[i][j] = hor[i][j - 1] + 1
            ver[i][j] = ver[i - 1][j] + 1
            dia[i][j] = dia[i - 1][j - 1] + 1
            ant[i][j] = ant[i - 1][j + 1] + 1
            vals = (hor[i][j], ver[i][j], dia[i][j], ant[i][j])
            best = max(best, *vals)
            print(f"ô ({i - 1},{j - 1}) -> {vals}  best={best}")
    for name, tab in (("ngang", hor), ("dọc", ver), ("chéo chính", dia), ("chéo phụ", ant)):
        print(f"bảng {name} (bỏ viền):")
        for i in range(1, m + 1):
            print("   ", tab[i][1:n + 1])
    return best


res = trace_dp(ex1)
print("kết quả:", res)
assert res == 3

thứ tự cột: (ngang, dọc, chéo chính, chéo phụ)
ô (0,1) -> (1, 1, 1, 1)  best=1
ô (0,2) -> (2, 1, 1, 1)  best=2
ô (1,1) -> (1, 2, 1, 2)  best=2
ô (1,2) -> (2, 2, 2, 1)  best=2
ô (2,3) -> (1, 1, 3, 1)  best=3
bảng ngang (bỏ viền):
    [0, 1, 2, 0]
    [0, 1, 2, 0]
    [0, 0, 0, 1]
bảng dọc (bỏ viền):
    [0, 1, 1, 0]
    [0, 2, 2, 0]
    [0, 0, 0, 1]
bảng chéo chính (bỏ viền):
    [0, 1, 1, 0]
    [0, 1, 2, 0]
    [0, 0, 0, 3]
bảng chéo phụ (bỏ viền):
    [0, 1, 1, 0]
    [0, 2, 1, 0]
    [0, 0, 0, 1]
kết quả: 3


Trace thêm ca chỉ có **chéo phụ** (dễ bị quên): ma trận `3 x 3` với đường `(0,2),(1,1),(2,0)`.

In [14]:
anti_only = [[0, 0, 1],
             [0, 1, 0],
             [1, 0, 0]]
res = trace_dp(anti_only)
print("kết quả:", res)
assert res == 3 == line_brute(anti_only)

thứ tự cột: (ngang, dọc, chéo chính, chéo phụ)
ô (0,2) -> (1, 1, 1, 1)  best=1
ô (1,1) -> (1, 1, 1, 2)  best=2
ô (2,0) -> (1, 1, 1, 3)  best=3
bảng ngang (bỏ viền):
    [0, 0, 1]
    [0, 1, 0]
    [1, 0, 0]
bảng dọc (bỏ viền):
    [0, 0, 1]
    [0, 1, 0]
    [1, 0, 0]
bảng chéo chính (bỏ viền):
    [0, 0, 1]
    [0, 1, 0]
    [1, 0, 0]
bảng chéo phụ (bỏ viền):
    [0, 0, 1]
    [0, 2, 0]
    [3, 0, 0]
kết quả: 3


## 9. Code nộp LeetCode

Đây là cách DP bốn bảng (cách 3) — ngắn, rõ, `O(mn)`. Nếu cần tiết kiệm bộ nhớ hãy thay bằng cách 4.

```python
from typing import List

class Solution:
    def longestLine(self, mat: List[List[int]]) -> int:
        m, n = len(mat), len(mat[0])
        hor = [[0] * (n + 2) for _ in range(m + 2)]   # ngang
        ver = [[0] * (n + 2) for _ in range(m + 2)]   # dọc
        dia = [[0] * (n + 2) for _ in range(m + 2)]   # chéo chính
        ant = [[0] * (n + 2) for _ in range(m + 2)]   # chéo phụ
        best = 0
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if mat[i - 1][j - 1]:
                    hor[i][j] = hor[i][j - 1] + 1
                    ver[i][j] = ver[i - 1][j] + 1
                    dia[i][j] = dia[i - 1][j - 1] + 1
                    ant[i][j] = ant[i - 1][j + 1] + 1
                    best = max(best, hor[i][j], ver[i][j], dia[i][j], ant[i][j])
        return best
```

In [15]:
class Solution:
    def longestLine(self, mat: List[List[int]]) -> int:
        m, n = len(mat), len(mat[0])
        hor = [[0] * (n + 2) for _ in range(m + 2)]
        ver = [[0] * (n + 2) for _ in range(m + 2)]
        dia = [[0] * (n + 2) for _ in range(m + 2)]
        ant = [[0] * (n + 2) for _ in range(m + 2)]
        best = 0
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if mat[i - 1][j - 1]:
                    hor[i][j] = hor[i][j - 1] + 1
                    ver[i][j] = ver[i - 1][j] + 1
                    dia[i][j] = dia[i - 1][j - 1] + 1
                    ant[i][j] = ant[i - 1][j + 1] + 1
                    best = max(best, hor[i][j], ver[i][j], dia[i][j], ant[i][j])
        return best


sol = Solution()
for mat, expected in examples:
    got = sol.longestLine(mat)
    print(got, "| kỳ vọng", expected)
    assert got == expected

3 | kỳ vọng 3
4 | kỳ vọng 4


## 10. Vì sao đúng

**Phát biểu.** Với mỗi hướng `d` và mỗi ô `(i, j)`, giá trị `f_d[i][j]` sau khi tính bằng độ dài chính xác của **dãy 1 liên tiếp dài nhất kết thúc tại `(i, j)`** theo hướng `d` (đọc từ trên/trái xuống).

**Chứng minh bằng quy nạp theo thứ tự duyệt (hàng trên xuống, cột trái sang phải).**

1. *Cơ sở:* ô ngoài biên (viền) có `f = 0` — đúng vì không có ô nào để tạo dãy.
2. *Bước quy nạp:* giả sử mọi ô đứng trước `(i, j)` theo 4 hướng đã đúng. Chúng nằm ở hàng `i` (cột nhỏ hơn) hoặc hàng `i-1`, nên đã được duyệt trước `(i, j)` — kể cả `(i-1, j+1)` vì nó ở hàng trên.
   - Nếu `mat[i][j] = 0` thì không có dãy 1 nào kết thúc tại đây: `f = 0` (đúng, vì bảng khởi tạo 0 và ta không ghi).
   - Nếu `mat[i][j] = 1`: một dãy 1 kết thúc tại `(i, j)` theo hướng `d` hoặc chỉ gồm riêng `(i, j)` (độ dài 1) hoặc là dãy kết thúc tại ô đứng trước kéo thêm `(i, j)`. Dãy dài nhất là `f_d[ô đứng trước] + 1`, trong đó trường hợp ô đứng trước là `0` hoặc ngoài biên cho `0 + 1 = 1`. Đúng.
3. Mọi đường thẳng gồm toàn số 1 có một ô cuối cùng (theo chiều duyệt); tại ô đó `f_d` bằng độ dài toàn đường. Vậy `max` của mọi `f_d` bằng độ dài đường dài nhất.

**Vì sao chỉ cần một chiều?** Một đường có độ dài `L` đọc xuôi hay ngược vẫn dài `L`. Mỗi hướng ta chỉ xét chiều "đi xuống/sang phải" nhưng vẫn chạm hết mọi đường thẳng.

**Vì sao không bỏ sót đường chéo phụ?** Bước đi `(1, -1)` ứng với ô đứng trước `(i-1, j+1)`, được xử lý bởi bảng `ant`.

## 11. Time/Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Từ mọi ô × 4 hướng | `O(m*n*max(m,n))` | `O(1)` | oracle, chậm với dãy dài |
| 2. Chỉ bắt đầu ở đầu dãy | `O(m*n)` | `O(1)` | không cần bảng |
| 3. DP bốn bảng | `O(m*n)` | `O(m*n)` | **cách nộp**, rõ ràng |
| 4. DP cuộn hàng | `O(m*n)` | `O(n)` | tiết kiệm bộ nhớ |
| 5. Gom theo đường + split | `O(m*n)` | `O(m*n)` | dễ chứng minh, độc lập với DP |

Với `m*n <= 10^4`, mọi cách `O(m*n)` chạy trong chưa tới một giây. Hằng số: 4 hướng nên cỡ `4 * 10^4` bước — rất nhỏ.

Lưu ý bộ nhớ cách 3: bốn bảng `(m+2) x (n+2)`. Với `1 x 10000` mỗi bảng là `3 x 10002` ô, với `100 x 100` là `102 x 102` — đều chỉ cỡ vài chục nghìn ô, nên không cần tối ưu thêm.

## 12. Bộ test

Chiến lược: (a) ví dụ của đề, (b) edge case (1 ô, 1 hàng, 1 cột, toàn 0, toàn 1, chỉ chéo phụ), (c) phản ví dụ cho lời giải thiếu hướng, (d) đối chiếu **mọi cách** với oracle `line_brute` trên rất nhiều ma trận ngẫu nhiên có seed, (e) ma trận lớn đúng ràng buộc `m * n <= 10^4`.

### 12.1. Ví dụ của đề và edge case

In [16]:
ALL = {
    "run_start": line_run_start,
    "dp4": line_dp4,
    "rolling": line_rolling,
    "group": line_group,
    "solution": lambda mat: Solution().longestLine(mat),
}

cases = [
    (ex1, 3),
    (ex2, 4),
    ([[0]], 0),
    ([[1]], 1),
    ([[1, 1, 1]], 3),                                   # một hàng
    ([[1], [1], [0], [1]], 2),                          # một cột, bị ngắt bởi 0
    ([[0, 0, 0], [0, 0, 0]], 0),                        # toàn 0
    ([[1] * 4 for _ in range(3)], 4),                   # toàn 1, 3x4 -> max(m, n)
    ([[1] * 3 for _ in range(4)], 4),                   # toàn 1, 4x3
    ([[1] * 5 for _ in range(5)], 5),                   # vuông toàn 1
    ([[0, 0, 1], [0, 1, 0], [1, 0, 0]], 3),             # chỉ chéo phụ
    ([[1, 0, 0], [0, 1, 0], [0, 0, 1]], 3),             # chỉ chéo chính
    ([[1, 0, 1, 1]], 2),                                # bị ngắt giữa chừng
    ([[1, 1, 0], [0, 1, 1], [1, 0, 1]], 3),             # chéo chính dài hơn hàng/cột
    ([[1, 0, 0], [0, 0, 0], [0, 0, 1]], 1),             # hai ô 1 cách xa trên cùng chéo
    ([[1, 0, 1], [0, 1, 0], [1, 0, 1]], 3),             # hình chữ X: tâm là 1 nên cả hai chéo đều dài 3
]

for mat, expected in cases:
    assert line_brute(mat) == expected, (mat, line_brute(mat), expected)
    for name, fn in ALL.items():
        got = fn(mat)
        assert got == expected, (name, mat, got, expected)
print("tất cả", len(cases), "ca cố định đều đúng với", len(ALL), "cách")

tất cả 16 ca cố định đều đúng với 5 cách


### 12.2. Phản ví dụ: lời giải thiếu một hướng hoặc nhầm hướng

Hàm dưới đây bỏ quên **chéo phụ** — rất dễ mắc vì ta hay nghĩ "ngang, dọc, chéo" là đủ. Ma trận chỉ có chéo phụ làm lộ lỗi.

In [17]:
def wrong_three_dirs(mat):
    m, n = len(mat), len(mat[0])
    hor = [[0] * (n + 2) for _ in range(m + 2)]
    ver = [[0] * (n + 2) for _ in range(m + 2)]
    dia = [[0] * (n + 2) for _ in range(m + 2)]
    best = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if mat[i - 1][j - 1]:
                hor[i][j] = hor[i][j - 1] + 1
                ver[i][j] = ver[i - 1][j] + 1
                dia[i][j] = dia[i - 1][j - 1] + 1
                best = max(best, hor[i][j], ver[i][j], dia[i][j])
    return best


anti_only = [[0, 0, 1], [0, 1, 0], [1, 0, 0]]
print("thiếu chéo phụ:", wrong_three_dirs(anti_only), "| đúng:", line_dp4(anti_only))
assert wrong_three_dirs(anti_only) == 1 and line_dp4(anti_only) == 3

# lời giải "đếm tổng số ô 1 trên hàng/cột" cũng sai: không đòi hỏi liên tiếp
def wrong_count_ones(mat):
    return max(max(sum(r) for r in mat), max(sum(c) for c in zip(*mat)))

gap = [[1, 0, 1, 1]]
print("đếm số 1 (không liên tiếp):", wrong_count_ones(gap), "| đúng:", line_dp4(gap))
assert wrong_count_ones(gap) == 3 and line_dp4(gap) == 2

thiếu chéo phụ: 1 | đúng: 3
đếm số 1 (không liên tiếp): 3 | đúng: 2


### 12.3. Đối chiếu ngẫu nhiên có seed (ma trận nhỏ, mật độ số 1 khác nhau)

Ma trận nhỏ và mật độ khác nhau (`p` từ thưa tới dày) giúp tạo cả ca nhiều ngắt quãng lẫn ca nhiều đường dài.

In [18]:
rng = random.Random(562)
checked = 0
for _ in range(1500):
    m = rng.randint(1, 8)
    n = rng.randint(1, 8)
    p = rng.choice([0.1, 0.3, 0.5, 0.7, 0.9, 1.0])
    mat = [[1 if rng.random() < p else 0 for _ in range(n)] for _ in range(m)]
    expected = line_brute(mat)
    for name, fn in ALL.items():
        assert fn(mat) == expected, (name, mat, fn(mat), expected)
    checked += 1
print("đã đối chiếu", checked, "ma trận ngẫu nhiên nhỏ")

đã đối chiếu 1500 ma trận ngẫu nhiên nhỏ


### 12.4. Ngẫu nhiên với hình dạng lệch (đúng ràng buộc `m * n <= 10^4`)

Chọn `m` bất kỳ rồi chọn `n <= 10^4 // m` — gồm cả những hình rất dài-mỏng như `2 x 3000`. Giới hạn kích thước để oracle brute không quá chậm.

In [19]:
rng = random.Random(2025)
count = 0
for _ in range(120):
    m = rng.choice([1, 2, 3, 5, 10, 30, 60, 100])
    n = rng.randint(1, min(10 ** 4 // m, 150))
    p = rng.choice([0.5, 0.8, 0.95])
    mat = [[1 if rng.random() < p else 0 for _ in range(n)] for _ in range(m)]
    expected = line_brute(mat)
    for name, fn in ALL.items():
        assert fn(mat) == expected, (name, m, n)
    count += 1
print("120 ma trận với hình dạng lệch đều khớp")

120 ma trận với hình dạng lệch đều khớp


### 12.5. Test kích thước lớn (đúng ràng buộc)

Chỉ chạy các cách `O(m*n)`. Với ma trận toàn 1, đáp án phải là `max(m, n)` (đường ngang hoặc dọc dài nhất; chéo chỉ dài `min(m, n)`).

In [20]:
import time

t = time.perf_counter()
big_cases = [(1, 10000), (10000, 1), (100, 100), (2, 5000), (5000, 2), (50, 200)]
for m, n in big_cases:
    ones = [[1] * n for _ in range(m)]
    answers = {name: fn(ones) for name, fn in ALL.items()}
    assert set(answers.values()) == {max(m, n)}, (m, n, answers)
    print(f"toàn 1 kích thước {m}x{n}: đáp án {max(m, n)} khớp mọi cách")

rng = random.Random(77)
for m, n in [(100, 100), (1, 10000), (10000, 1), (40, 250)]:
    mat = [[1 if rng.random() < 0.9 else 0 for _ in range(n)] for _ in range(m)]
    answers = {name: fn(mat) for name, fn in ALL.items()}
    assert len(set(answers.values())) == 1, (m, n, answers)
    print(f"ngẫu nhiên 90% là 1, {m}x{n}: đáp án {answers['dp4']}")
print("thời gian các test lớn:", round(time.perf_counter() - t, 2), "s")

toàn 1 kích thước 1x10000: đáp án 10000 khớp mọi cách
toàn 1 kích thước 10000x1: đáp án 10000 khớp mọi cách
toàn 1 kích thước 100x100: đáp án 100 khớp mọi cách
toàn 1 kích thước 2x5000: đáp án 5000 khớp mọi cách


toàn 1 kích thước 5000x2: đáp án 5000 khớp mọi cách
toàn 1 kích thước 50x200: đáp án 200 khớp mọi cách
ngẫu nhiên 90% là 1, 100x100: đáp án 66


ngẫu nhiên 90% là 1, 1x10000: đáp án 53
ngẫu nhiên 90% là 1, 10000x1: đáp án 81
ngẫu nhiên 90% là 1, 40x250: đáp án 74
thời gian các test lớn: 0.35 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên đường chéo phụ

Xem `wrong_three_dirs` ở mục 12.2. **Vì sao sai:** đề nói "horizontal, vertical, diagonal, **anti-diagonal**" — bốn hướng. Nhiều người chỉ nghĩ ba hướng quen thuộc. **Code đúng:** thêm bảng `ant` với ô đứng trước là `(i-1, j+1)`.

### Lỗi 2 — Không có viền: chỉ số âm "quay vòng" và `IndexError`

Viết DP trực tiếp trên bảng cùng kích thước `m x n` rồi đọc `dp[i-1][j-1]` ở `j = 0`.

In [21]:
def wrong_wrap(mat):
    """Chéo chính không có viền: dia[i-1][j-1] ở j = 0 đọc dia[i-1][-1] (cột cuối!)."""
    m, n = len(mat), len(mat[0])
    dia = [[0] * n for _ in range(m)]
    best = 0
    for i in range(m):
        for j in range(n):
            if mat[i][j]:
                dia[i][j] = dia[i - 1][j - 1] + 1
                best = max(best, dia[i][j])
    return best


def wrong_anti_index(mat):
    """Chéo phụ không có viền: ant[i-1][j+1] ở j = n-1 vượt biên."""
    m, n = len(mat), len(mat[0])
    ant = [[0] * n for _ in range(m)]
    best = 0
    for i in range(m):
        for j in range(n):
            if mat[i][j]:
                ant[i][j] = ant[i - 1][j + 1] + 1
                best = max(best, ant[i][j])
    return best


tricky = [[0, 0, 1],
          [1, 0, 0]]
print("quay vòng:", wrong_wrap(tricky), "| đúng:", line_dp4(tricky))
assert wrong_wrap(tricky) == 2 and line_dp4(tricky) == 1
try:
    wrong_anti_index([[0, 1], [1, 1]])
except IndexError as e:
    print("IndexError:", e)

quay vòng: 2 | đúng: 1
IndexError: list index out of range


**Vì sao sai:** ô `(1,0)` đáng lẽ không có ô đứng trước theo chéo chính; nhưng `dia[0][-1]` lại là `dia[0][2]` (ô cuối hàng 0, đang là 1), nên cộng nhầm ra `2`. Lỗi này **không báo gì**. **Cách đúng:** thêm viền 0 (kích thước `(m+2) x (n+2)`, ô gốc ở `[i+1][j+1]`) hoặc kiểm tra biên bằng `if`.

### Lỗi 3 — Quên reset khi gặp số 0

In [22]:
def wrong_no_reset(mat):
    m, n = len(mat), len(mat[0])
    best = 0
    for i in range(m):
        left = 0
        for j in range(n):
            if mat[i][j]:
                left += 1
                best = max(best, left)
            # SAI: thiếu 'else: left = 0'
    return best


print("sai :", wrong_no_reset([[1, 0, 1, 1]]), "| đúng:", line_dp4([[1, 0, 1, 1]]))
assert wrong_no_reset([[1, 0, 1, 1]]) == 3 and line_dp4([[1, 0, 1, 1]]) == 2

sai : 3 | đúng: 2


**Vì sao sai:** số 0 phải **cắt** dãy; biến đếm cần về 0. Ở DP bốn bảng, việc này tự nhiên vì ô 0 không được ghi (giữ 0); ở bản cuộn hàng với biến `left` thì phải reset thủ công.

### Lỗi 4 — Tạo bảng 2D bằng phép nhân `[[0] * n] * m`

In [23]:
def wrong_alias(mat):
    m, n = len(mat), len(mat[0])
    hor = [[0] * (n + 2)] * (m + 2)      # SAI: mọi hàng là CÙNG một list
    ver = [[0] * (n + 2)] * (m + 2)
    dia = [[0] * (n + 2)] * (m + 2)
    ant = [[0] * (n + 2)] * (m + 2)
    best = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if mat[i - 1][j - 1]:
                hor[i][j] = hor[i][j - 1] + 1
                ver[i][j] = ver[i - 1][j] + 1
                dia[i][j] = dia[i - 1][j - 1] + 1
                ant[i][j] = ant[i - 1][j + 1] + 1
                best = max(best, hor[i][j], ver[i][j], dia[i][j], ant[i][j])
    return best


col = [[1], [0], [1]]
print("sai :", wrong_alias(col), "| đúng:", line_dp4(col))
assert wrong_alias(col) == 2 and line_dp4(col) == 1

sai : 2 | đúng: 1


**Vì sao sai:** ghi vào `ver[3][1]` đồng thời ghi vào mọi hàng, nên số 1 ở hàng 3 "nhìn thấy" số 1 ở hàng 1 dù giữa chúng có số 0. **Cách đúng:** `[[0] * (n + 2) for _ in range(m + 2)]`.

### Lỗi 5 — Nhầm `m` (số hàng) và `n` (số cột)

In [24]:
def wrong_swap_mn(mat):
    n, m = len(mat), len(mat[0])          # SAI: bị đảo
    best = 0
    for i in range(m):
        for j in range(n):
            if mat[i][j]:
                best = max(best, 1)
    return best


try:
    wrong_swap_mn([[1, 1, 1, 1]])         # ma trận 1x4
except IndexError as e:
    print("IndexError khi m, n bị đảo:", e)

IndexError khi m, n bị đảo: list index out of range


**Cách đúng:** `m, n = len(mat), len(mat[0])` — `len(mat)` là số **hàng**, `len(mat[0])` là số **cột**. Test luôn bằng ma trận **không vuông** (`1 x 4`, `4 x 1`) để lộ lỗi này; ma trận vuông che giấu nó.

## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra "DP theo hướng trên ma trận":**

- Hỏi **dãy liên tiếp dài nhất** trên lưới theo vài hướng cố định → mỗi hướng một bảng "độ dài kết thúc tại ô này".
- Giá trị ô hiện tại phụ thuộc **đúng một ô đứng trước** theo hướng đó → `f[i][j] = f[trước] + 1` (nếu thỏa), ngược lại `0` hoặc `1`.

**Mẹo nhớ:**

1. Bảng chữ: `(0, 1)` ngang, `(1, 0)` dọc, `(1, 1)` chéo chính, `(1, -1)` chéo phụ; ô đứng trước = `(i - di, j - dj)`.
2. Duyệt hàng từ trên xuống, trái sang phải để mọi ô đứng trước (kể cả `(i-1, j+1)`) đã sẵn sàng.
3. **Viền 0** loại bỏ mọi `if` kiểm tra biên và cả bẫy chỉ số âm của Python.
4. Muốn tiết kiệm bộ nhớ: chỉ cần **hàng trước** + một biến cho ngang → `O(n)`.
5. Cần đối chiếu: cách gom đường thẳng + `split('0')` đơn giản tới mức khó sai.
6. Nếu đề đổi "toàn 1" thành "toàn cùng giá trị", chỉ cần đổi điều kiện `mat[i][j] == 1` thành `mat[i][j] == mat[ô trước]` (xem bài 5 mục 17).
7. Các bài cùng họ: LeetCode 485 (Max Consecutive Ones, 1 chiều), 221 (Maximal Square), 1139 (Largest 1-Bordered Square), 1504 (Count Submatrices With All Ones).

## 15. Biến thể

### 15.1. Độ dài lớn nhất **theo từng hướng**

Tách riêng bốn đáp án rồi lấy `max` — tiện để kiểm tra từng bảng.

In [25]:
def longest_per_direction(mat):
    m, n = len(mat), len(mat[0])
    best = {"ngang": 0, "dọc": 0, "chéo chính": 0, "chéo phụ": 0}
    hor = [[0] * (n + 2) for _ in range(m + 2)]
    ver = [[0] * (n + 2) for _ in range(m + 2)]
    dia = [[0] * (n + 2) for _ in range(m + 2)]
    ant = [[0] * (n + 2) for _ in range(m + 2)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if mat[i - 1][j - 1]:
                hor[i][j] = hor[i][j - 1] + 1
                ver[i][j] = ver[i - 1][j] + 1
                dia[i][j] = dia[i - 1][j - 1] + 1
                ant[i][j] = ant[i - 1][j + 1] + 1
                best["ngang"] = max(best["ngang"], hor[i][j])
                best["dọc"] = max(best["dọc"], ver[i][j])
                best["chéo chính"] = max(best["chéo chính"], dia[i][j])
                best["chéo phụ"] = max(best["chéo phụ"], ant[i][j])
    return best


print(longest_per_direction(ex1))
print(longest_per_direction(ex2))
assert max(longest_per_direction(ex1).values()) == 3
assert max(longest_per_direction(ex2).values()) == 4

{'ngang': 2, 'dọc': 2, 'chéo chính': 3, 'chéo phụ': 2}
{'ngang': 4, 'dọc': 2, 'chéo chính': 3, 'chéo phụ': 2}


### 15.2. Trả về **vị trí** hai đầu mút của đường dài nhất

Dùng cách 2 (đi từ đầu dãy) và ghi lại đầu mút cuối.

In [26]:
def longest_with_ends(mat):
    m, n = len(mat), len(mat[0])
    best = (0, None, None)
    for i in range(m):
        for j in range(n):
            if mat[i][j] != 1:
                continue
            for di, dj in DIRS:
                px, py = i - di, j - dj
                if 0 <= px < m and 0 <= py < n and mat[px][py] == 1:
                    continue
                length, x, y = 0, i, j
                while 0 <= x < m and 0 <= y < n and mat[x][y] == 1:
                    length += 1
                    x += di
                    y += dj
                if length > best[0]:
                    best = (length, (i, j), (x - di, y - dj))
    return best


length, start, end = longest_with_ends(ex1)
print("dài", length, "từ", start, "đến", end)
assert length == 3 and start == (0, 1) and end == (2, 3)

# kiểm chứng ngẫu nhiên: các ô giữa hai đầu mút đều là 1
rng = random.Random(15)
for _ in range(300):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    mat = [[1 if rng.random() < 0.7 else 0 for _ in range(n)] for _ in range(m)]
    length, s, e = longest_with_ends(mat)
    assert length == line_dp4(mat)
    if length:
        di = (e[0] - s[0]) // max(length - 1, 1)
        dj = (e[1] - s[1]) // max(length - 1, 1)
        assert all(mat[s[0] + t * di][s[1] + t * dj] == 1 for t in range(length))
print("15.2 OK")

dài 3 từ (0, 1) đến (2, 3)
15.2 OK


### 15.3. Chỉ xét 1 chiều: `Max Consecutive Ones` (LeetCode 485)

Bài gốc 1 chiều, chính là phép tính ta làm trên **mỗi đường thẳng**. Dùng `itertools.groupby` để gom cụm các giá trị giống nhau liên tiếp.

In [27]:
def max_consecutive_ones(nums):
    return max((len(list(g)) for v, g in groupby(nums) if v == 1), default=0)


assert max_consecutive_ones([1, 1, 0, 1, 1, 1]) == 3
assert max_consecutive_ones([1, 0, 1, 1, 0, 1]) == 2
assert max_consecutive_ones([0, 0]) == 0
print(max_consecutive_ones([1, 1, 0, 1, 1, 1]))

3


### 15.4. Đếm số đường thẳng **cực đại** có độ dài `>= L`

Một dãy là **cực đại** nếu không thể kéo dài thêm ở hai đầu. Dùng cách 2: mỗi dãy được đếm đúng **một lần** tại đầu dãy.

In [28]:
def count_maximal_lines(mat, L):
    m, n = len(mat), len(mat[0])
    total = 0
    for i in range(m):
        for j in range(n):
            if mat[i][j] != 1:
                continue
            for di, dj in DIRS:
                px, py = i - di, j - dj
                if 0 <= px < m and 0 <= py < n and mat[px][py] == 1:
                    continue
                length, x, y = 0, i, j
                while 0 <= x < m and 0 <= y < n and mat[x][y] == 1:
                    length += 1
                    x += di
                    y += dj
                if length >= L:
                    total += 1
    return total


print(count_maximal_lines(ex1, 2), count_maximal_lines(ex1, 3), count_maximal_lines(ex1, 4))
assert count_maximal_lines(ex1, 3) == 1
assert count_maximal_lines(ex1, 4) == 0
assert count_maximal_lines(ex1, 1) >= count_maximal_lines(ex1, 2)

6 1 0


### 15.5. Họ hàng: hình vuông lớn nhất toàn số 1 (LeetCode 221)

Cũng là DP trên ma trận nhưng trạng thái là "cạnh hình vuông lớn nhất có góc dưới-phải tại ô này", công thức `min` của ba ô lân cận cộng 1. Để thấy khác biệt: bài 562 dùng **một** ô đứng trước cho mỗi hướng; bài 221 dùng **ba** ô.

In [29]:
def maximal_square(mat):
    m, n = len(mat), len(mat[0])
    f = [[0] * (n + 1) for _ in range(m + 1)]
    best = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if mat[i - 1][j - 1]:
                f[i][j] = min(f[i - 1][j], f[i][j - 1], f[i - 1][j - 1]) + 1
                best = max(best, f[i][j])
    return best * best


def maximal_square_oracle(mat):
    m, n = len(mat), len(mat[0])
    best = 0
    for i in range(m):
        for j in range(n):
            for s in range(1, min(m - i, n - j) + 1):
                if all(mat[x][y] for x in range(i, i + s) for y in range(j, j + s)):
                    best = max(best, s * s)
    return best


rng = random.Random(221)
for _ in range(200):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    mat = [[1 if rng.random() < 0.7 else 0 for _ in range(n)] for _ in range(m)]
    assert maximal_square(mat) == maximal_square_oracle(mat)
print("221 khớp oracle; ví dụ:", maximal_square([[1, 0, 1, 0, 0], [1, 0, 1, 1, 1], [1, 1, 1, 1, 1], [1, 0, 0, 1, 0]]))

221 khớp oracle; ví dụ: 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Từ mọi ô × 4 hướng | đi thẳng từ mỗi ô 1 | `O(m*n*max(m,n))` | `O(1)` | oracle, ma trận rất nhỏ |
| 2. Bắt đầu ở đầu dãy | chỉ đi khi ô đứng trước không phải 1 | `O(m*n)` | `O(1)` | khi muốn không dùng bảng; cũng trả được đầu mút |
| 3. DP bốn bảng | `f = f[trước] + 1` | `O(m*n)` | `O(m*n)` | **cách nộp**, dễ đọc, dễ mở rộng |
| 4. DP cuộn hàng | chỉ nhớ hàng trước + biến `left` | `O(m*n)` | `O(n)` | khi giới hạn bộ nhớ |
| 5. Gom đường + `split('0')` | mỗi đường là một chuỗi | `O(m*n)` | `O(m*n)` | đối chiếu, dễ chứng minh đúng |

Khuyến nghị: nộp **cách 3**; dùng cách 1 làm oracle khi tự mở rộng; dùng cách 5 làm oracle thứ hai vì nó **độc lập hoàn toàn với DP**.

In [30]:
rng = random.Random(16)
for _ in range(400):
    m, n = rng.randint(1, 9), rng.randint(1, 9)
    mat = [[1 if rng.random() < rng.choice([0.3, 0.6, 0.9]) else 0 for _ in range(n)] for _ in range(m)]
    res = {name: fn(mat) for name, fn in ALL.items()}
    res["brute"] = line_brute(mat)
    assert len(set(res.values())) == 1, (mat, res)
print("400 ma trận: 6 hàm cho cùng đáp án")

400 ma trận: 6 hàm cho cùng đáp án


## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
print([(i - j, i + j) for i, j in [(0, 0), (1, 2), (2, 1)]])
print(line_dp4([[1, 1], [1, 1]]))
print(line_dp4([[0, 1, 1, 1]]))
```

In [31]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay bảng DP (dễ-vừa)

Với ma trận

```
1 1 0
0 1 1
1 0 1
```

hãy tính tay: (a) đáp án `longestLine`; (b) giá trị bảng **chéo chính** `dia` tại ô gốc `(2,2)` (hàng 2, cột 2, đếm từ 0); (c) giá trị bảng **chéo phụ** `ant` tại ô gốc `(1,1)`.

In [32]:
# Bài 2: điền kết quả tính tay
hand_answer = None
hand_dia_at_2_2 = None
hand_ant_at_1_1 = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành bản DP **cuộn hàng**.

In [33]:
def exercise_3_rolling(mat: List[List[int]]) -> int:
    m, n = len(mat), len(mat[0])
    up = [0] * (n + 2)
    dg = [0] * (n + 2)
    ad = [0] * (n + 2)
    best = 0
    for i in range(m):
        cu = [0] * (n + 2)
        cd = [0] * (n + 2)
        ca = [0] * (n + 2)
        left = 0
        for j in range(1, n + 1):
            # TODO 1: nếu mat[i][j - 1] là 1 thì tăng left, tính cu[j], cd[j], ca[j] từ up/dg/ad
            # TODO 2: cập nhật best (đừng quên left)
            # TODO 3: nếu là 0 thì reset left
            pass
        # TODO 4: đẩy hàng hiện tại thành hàng trước
    return None


# assert exercise_3_rolling(ex1) == 3
# assert exercise_3_rolling(ex2) == 4
# assert exercise_3_rolling([[0, 0, 1], [0, 1, 0], [1, 0, 0]]) == 3

### Bài 4 — Tự viết lại bằng ý tưởng khác (vừa-khó)

Không nhìn lại cách 5, hãy tự viết `exercise_4_lines(mat)`: gom các ô theo mã đường (hàng `i`, cột `j`, `i - j`, `i + j`), nối thành chuỗi `'0'/'1'`, rồi dùng `split('0')`.

```python
exercise_4_lines(ex1) -> 3
exercise_4_lines([[1, 0, 1, 1]]) -> 2
exercise_4_lines([[0]]) -> 0
```

In [34]:
def exercise_4_lines(mat: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_4_lines(ex1) == 3
# assert exercise_4_lines([[1, 0, 1, 1]]) == 2
# assert exercise_4_lines([[0]]) == 0

### Bài 5 — Mini challenge: đường dài nhất gồm các giá trị **bằng nhau** (khó)

Ma trận `mat` chứa số nguyên **bất kỳ** (không chỉ 0/1). Viết `exercise_5_equal(mat)` trả về độ dài đường thẳng (ngang, dọc, chéo chính, chéo phụ) dài nhất mà **mọi ô có cùng một giá trị** (giá trị nào cũng được, kể cả `0`).

```python
exercise_5_equal([[1, 1, 2], [2, 2, 2], [1, 1, 1]]) -> 3
exercise_5_equal([[7]]) -> 1
exercise_5_equal([[1, 2], [3, 4]]) -> 1
exercise_5_equal([[0, 0], [0, 0]]) -> 2
```

Gợi ý hướng nghĩ: giống cách 3 nhưng điều kiện "ô hiện tại là 1" đổi thành "ô đứng trước tồn tại và có cùng giá trị với ô hiện tại"; độ dài tối thiểu luôn là `1`.

In [35]:
def exercise_5_equal(mat: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_5_equal([[1, 1, 2], [2, 2, 2], [1, 1, 1]]) == 3
# assert exercise_5_equal([[7]]) == 1
# assert exercise_5_equal([[1, 2], [3, 4]]) == 1
# assert exercise_5_equal([[0, 0], [0, 0]]) == 2

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `i - j` là mã chéo chính, `i + j` là mã chéo phụ.
- Mức 2: dòng 2: ma trận `2 x 2` toàn 1 có hàng dài 2, cột dài 2, chéo dài 2. Dòng 3: một hàng có ba số 1 liên tiếp ở cuối.
- Mức 3: dòng 1 là `[(0, 0), (-1, 3), (1, 3)]`; dòng 2 là `2`; dòng 3 là `3`.

**Bài 2**
- Mức 1: vẽ từng đường: hàng, cột, chéo chính `(0,0),(1,1),(2,2)`.
- Mức 2: chéo chính `(0,0),(1,1),(2,2)` toàn 1; chéo phụ qua `(1,1)` gồm `(0,2),(1,1),(2,0)` = `0,1,1`.
- Mức 3: (a) `3`; (b) `dia` tại `(2,2)` là `3`; (c) `ant` tại `(1,1)` = `ant` tại `(0,2)` + 1 = `0 + 1 = 1` (vì `(0,2)` là 0).

**Bài 3**
- Mức 1: khi ô là 1: `left += 1`, `cu[j] = up[j] + 1`, `cd[j] = dg[j - 1] + 1`, `ca[j] = ad[j + 1] + 1`.
- Mức 2: `best = max(best, left, cu[j], cd[j], ca[j])`; else nhánh `left = 0`.
- Mức 3: cuối mỗi hàng `up, dg, ad = cu, cd, ca`; cuối hàm `return best`.

**Bài 4**
- Mức 1: tạo `rows`, `cols` (list các list) và hai dict `diag`, `anti`.
- Mức 2: mỗi ô thêm ký tự `'1'`/`'0'` vào bốn nhóm theo thứ tự duyệt.
- Mức 3: `max(len(p) for p in "".join(g).split("0"))` cho mỗi nhóm; lấy `max` toàn bộ.

**Bài 5**
- Mức 1: có `f[i][j][d]` cho 4 hướng, khởi tạo mọi giá trị `1`.
- Mức 2: với mỗi hướng `(di, dj)`, ô trước là `(i - di, j - dj)`; nếu trong biên và cùng giá trị thì `f[i][j][d] = f[trước][d] + 1`.
- Mức 3: duyệt hàng từ trên xuống, trái sang phải; `best` khởi tạo `1`; dùng `DIRS` đã có sẵn trong notebook.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [36]:
print([(i - j, i + j) for i, j in [(0, 0), (1, 2), (2, 1)]])
print(line_dp4([[1, 1], [1, 1]]))
print(line_dp4([[0, 1, 1, 1]]))

assert [(i - j, i + j) for i, j in [(0, 0), (1, 2), (2, 1)]] == [(0, 0), (-1, 3), (1, 3)]
assert line_dp4([[1, 1], [1, 1]]) == 2
assert line_dp4([[0, 1, 1, 1]]) == 3

[(0, 0), (-1, 3), (1, 3)]
2
3


### Đáp án Bài 2

Ma trận `mat = [[1,1,0],[0,1,1],[1,0,1]]`:

- Ngang: hàng 0 dài 2, hàng 1 dài 2, hàng 2 dài 1.
- Dọc: cột 1 dài 2 (`1,1,0`), cột 2 dài 2 (`0,1,1`).
- Chéo chính `(0,0),(1,1),(2,2)` = `1,1,1` → **3**.
- Chéo phụ qua `(1,1)`: `(0,2),(1,1),(2,0)` = `0,1,1` → 2.

Đáp án (a) = 3; (b) `dia` tại `(2,2)` = 3; (c) `ant` tại `(1,1)`: ô trước là `(0,2)` có giá trị 0, nên `ant = 0 + 1 = 1`. (Còn `ant` tại `(2,0)` mới là 2.)

In [37]:
hand_mat = [[1, 1, 0], [0, 1, 1], [1, 0, 1]]
res = trace_dp(hand_mat)
assert res == 3 == line_brute(hand_mat)

# kiểm bảng chéo chính / chéo phụ bằng cách tính lại độc lập (đi ngược về đầu dãy)
def run_back(mat, i, j, di, dj):
    """độ dài dãy 1 kết thúc tại (i, j) theo hướng (di, dj), đi lùi từ (i, j)."""
    length = 0
    x, y = i, j
    while 0 <= x < len(mat) and 0 <= y < len(mat[0]) and mat[x][y] == 1:
        length += 1
        x, y = x - di, y - dj
    return length

assert run_back(hand_mat, 2, 2, 1, 1) == 3
assert run_back(hand_mat, 1, 1, 1, -1) == 1
assert run_back(hand_mat, 2, 0, 1, -1) == 2
print("Bài 2 OK")

thứ tự cột: (ngang, dọc, chéo chính, chéo phụ)
ô (0,0) -> (1, 1, 1, 1)  best=1
ô (0,1) -> (2, 1, 1, 1)  best=2
ô (1,1) -> (1, 2, 2, 1)  best=2
ô (1,2) -> (2, 1, 2, 1)  best=2
ô (2,0) -> (1, 1, 1, 2)  best=2
ô (2,2) -> (1, 2, 3, 1)  best=3
bảng ngang (bỏ viền):
    [1, 2, 0]
    [0, 1, 2]
    [1, 0, 1]
bảng dọc (bỏ viền):
    [1, 1, 0]
    [0, 2, 1]
    [1, 0, 2]
bảng chéo chính (bỏ viền):
    [1, 1, 0]
    [0, 2, 2]
    [1, 0, 3]
bảng chéo phụ (bỏ viền):
    [1, 1, 0]
    [0, 1, 1]
    [2, 0, 1]
Bài 2 OK


### Đáp án Bài 3

In [38]:
def exercise_3_rolling_solution(mat: List[List[int]]) -> int:
    m, n = len(mat), len(mat[0])
    up = [0] * (n + 2)
    dg = [0] * (n + 2)
    ad = [0] * (n + 2)
    best = 0
    for i in range(m):
        cu = [0] * (n + 2)
        cd = [0] * (n + 2)
        ca = [0] * (n + 2)
        left = 0
        for j in range(1, n + 1):
            if mat[i][j - 1]:
                left += 1
                cu[j] = up[j] + 1
                cd[j] = dg[j - 1] + 1
                ca[j] = ad[j + 1] + 1
                best = max(best, left, cu[j], cd[j], ca[j])
            else:
                left = 0
        up, dg, ad = cu, cd, ca
    return best


assert exercise_3_rolling_solution(ex1) == 3
assert exercise_3_rolling_solution(ex2) == 4
assert exercise_3_rolling_solution([[0, 0, 1], [0, 1, 0], [1, 0, 0]]) == 3
rng = random.Random(3)
for _ in range(300):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    mat = [[1 if rng.random() < 0.6 else 0 for _ in range(n)] for _ in range(m)]
    assert exercise_3_rolling_solution(mat) == line_brute(mat)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [39]:
def exercise_4_lines_solution(mat: List[List[int]]) -> int:
    m, n = len(mat), len(mat[0])
    groups = {}
    for i in range(m):
        for j in range(n):
            ch = "1" if mat[i][j] else "0"
            for key in (("r", i), ("c", j), ("d", i - j), ("a", i + j)):
                groups.setdefault(key, []).append(ch)
    best = 0
    for g in groups.values():
        best = max(best, max(len(p) for p in "".join(g).split("0")))
    return best


assert exercise_4_lines_solution(ex1) == 3
assert exercise_4_lines_solution([[1, 0, 1, 1]]) == 2
assert exercise_4_lines_solution([[0]]) == 0
rng = random.Random(4)
for _ in range(300):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    mat = [[1 if rng.random() < 0.6 else 0 for _ in range(n)] for _ in range(m)]
    assert exercise_4_lines_solution(mat) == line_brute(mat)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [40]:
def exercise_5_equal_solution(mat: List[List[int]]) -> int:
    m, n = len(mat), len(mat[0])
    f = [[[1] * 4 for _ in range(n)] for _ in range(m)]
    best = 1
    for i in range(m):
        for j in range(n):
            for d, (di, dj) in enumerate(DIRS):
                pi, pj = i - di, j - dj
                if 0 <= pi < m and 0 <= pj < n and mat[pi][pj] == mat[i][j]:
                    f[i][j][d] = f[pi][pj][d] + 1
                best = max(best, f[i][j][d])
    return best


def equal_oracle(mat):
    m, n = len(mat), len(mat[0])
    best = 1
    for i in range(m):
        for j in range(n):
            for di, dj in DIRS:
                length, x, y = 0, i, j
                while 0 <= x < m and 0 <= y < n and mat[x][y] == mat[i][j]:
                    length += 1
                    x += di
                    y += dj
                best = max(best, length)
    return best


assert exercise_5_equal_solution([[1, 1, 2], [2, 2, 2], [1, 1, 1]]) == 3
assert exercise_5_equal_solution([[7]]) == 1
assert exercise_5_equal_solution([[1, 2], [3, 4]]) == 1
assert exercise_5_equal_solution([[0, 0], [0, 0]]) == 2
rng = random.Random(5)
for _ in range(400):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    mat = [[rng.randint(0, 2) for _ in range(n)] for _ in range(m)]
    assert exercise_5_equal_solution(mat) == equal_oracle(mat)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
from typing import List

class Solution:
    def longestLine(self, mat: List[List[int]]) -> int:
        m, n = len(mat), len(mat[0])
        hor = [[0] * (n + 2) for _ in range(m + 2)]   # ngang    : ô trước (i, j-1)
        ver = [[0] * (n + 2) for _ in range(m + 2)]   # dọc      : ô trước (i-1, j)
        dia = [[0] * (n + 2) for _ in range(m + 2)]   # chéo chính: ô trước (i-1, j-1)
        ant = [[0] * (n + 2) for _ in range(m + 2)]   # chéo phụ : ô trước (i-1, j+1)
        best = 0
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if mat[i - 1][j - 1]:
                    hor[i][j] = hor[i][j - 1] + 1
                    ver[i][j] = ver[i - 1][j] + 1
                    dia[i][j] = dia[i - 1][j - 1] + 1
                    ant[i][j] = ant[i - 1][j + 1] + 1
                    best = max(best, hor[i][j], ver[i][j], dia[i][j], ant[i][j])
        return best
```

Câu thần chú: **"bốn hướng, bốn bảng, viền 0; ô là 1 thì lấy ô trước cộng 1, ô là 0 thì giữ 0; lấy max."**

### Checklist trước khi submit

- [ ] Đủ **4 hướng**, đặc biệt **chéo phụ** `(i-1, j+1)`.
- [ ] Bảng có viền `(m+2) x (n+2)`; ô gốc `mat[i-1][j-1]` ứng với bảng `[i][j]`.
- [ ] Tạo bảng bằng `[[0] * (n + 2) for _ in range(m + 2)]`, không dùng `[[0] * n] * m`.
- [ ] Ô 0 không bị ghi (hoặc reset `left = 0` nếu dùng biến cuộn hàng).
- [ ] `m, n = len(mat), len(mat[0])`; đã test ma trận không vuông (`1 x 4`, `4 x 1`).
- [ ] Đã test: toàn 0, toàn 1 (đáp án `max(m, n)`), chỉ chéo phụ, một ô.
- [ ] Skeleton đúng: `class Solution`, method `longestLine(self, mat: List[List[int]]) -> int`; độ phức tạp `O(m*n)` thời gian.
- [ ] Bài Premium: nếu có tài khoản, đối chiếu lại đề gốc và ràng buộc.